# Bài 9 · PCA và ảnh thời trang · Thực hành trên lớp (v3)

Mở notebook từ thư mục bài. Dữ liệu: `data/fashion_subset.npz` (Fashion-MNIST, Zalando Research, giấy phép MIT): 12.000 ảnh train, 3.000 ảnh validation; 10.000 ảnh test **không được đọc** trong bài này (để dành cho Bài 10).
Mã cho sẵn; việc của bạn là chạy từng ô, đối chiếu với slide và trả lời câu hỏi in đậm. Ô có dấu ✅ là ô tự kiểm.

In [1]:
from pathlib import Path
import time
import numpy as np
from fractions import Fraction as Fr
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
ROOT = Path.cwd()
with np.load(ROOT / 'data' / 'fashion_subset.npz') as d:
    X = d['train_images'].astype(np.float64) / 255; y = d['train_labels']
    V = d['validation_images'].astype(np.float64) / 255; yv = d['validation_labels']
TEN = ['áo phông', 'quần dài', 'áo len', 'váy', 'áo khoác', 'dép xăng-đan', 'áo sơ mi', 'giày thể thao', 'túi', 'bốt cổ ngắn']
print(X.shape, V.shape, np.bincount(y))

(12000, 784) (3000, 784) [1200 1200 1200 1200 1200 1200 1200 1200 1200 1200]


## Chặng 1 · Ảnh là một điểm trong 784 chiều (slide 5–16)

Pixel ở hàng r, cột c là phần tử thứ 28·r + c của vector. Đo tương quan của hai pixel kề nhau và hai pixel ở hai góc xa nhau (trên train).

In [2]:
p = lambda r, c: 28 * r + c
r_ke = np.corrcoef(X[:, p(14, 14)], X[:, p(15, 14)])[0, 1]
r_xa = np.corrcoef(X[:, p(5, 5)], X[:, p(22, 22)])[0, 1]
mu = X.mean(0)                     # fit trên train
Xc = X - mu; Vc = V - mu           # validation dùng lại đúng μ của train
print('r kề nhau =', round(r_ke, 3), '· r xa nhau =', round(r_xa, 3))

r kề nhau = 0.883 · r xa nhau = 0.172


In [3]:
# ✅ TỰ KIỂM · chặng 1
assert abs(r_ke - 0.883013) < 1e-6 and abs(r_xa - 0.172149) < 1e-6
assert np.allclose(Xc.mean(0), 0) and not np.allclose(Vc.mean(0), 0)
print('✅ tương quan đúng slide 11; trung bình train đã về 0, validation thì không (vì dùng μ của train)')

✅ tương quan đúng slide 11; trung bình train đã về 0, validation thì không (vì dùng μ của train)


**Câu hỏi:** vì sao trung bình của `Vc` không bằng 0? Có sai không?

## Chặng 2 · Một hướng tốt nhất trên bốn điểm (slide 17–32)

Bốn điểm A(3; 4), B(4; 3), C(6; 7), D(7; 6). Tính bằng phân số để thấy các số 9/2, 1/2, 5.

In [4]:
P = np.array([[3, 4], [4, 3], [6, 7], [7, 6]], float)
Pc = P - P.mean(0)
def giu_bo(theta_deg):
    w = np.array([np.cos(np.radians(theta_deg)), np.sin(np.radians(theta_deg))])
    z = Pc @ w                                  # score
    rec = np.outer(z, w)                        # tái tạo (đã căn giữa)
    return np.mean(z ** 2), np.mean(np.sum((Pc - rec) ** 2, 1))
for th in [0, 20, 45, 70, 135]:
    v, e = giu_bo(th); print(f'θ = {th:3d}°: giữ {v:.3f} · bỏ {e:.3f} · tổng {v + e:.3f}')
C2 = Pc.T @ Pc / len(P)                         # chia n như slide 28
lam2, W2 = np.linalg.eigh(C2)
print('C =', C2.tolist(), '· trị riêng (tăng dần):', lam2)

θ =   0°: giữ 2.500 · bỏ 2.500 · tổng 5.000
θ =  20°: giữ 3.786 · bỏ 1.214 · tổng 5.000
θ =  45°: giữ 4.500 · bỏ 0.500 · tổng 5.000
θ =  70°: giữ 3.786 · bỏ 1.214 · tổng 5.000
θ = 135°: giữ 0.500 · bỏ 4.500 · tổng 5.000
C = [[2.5, 2.0], [2.0, 2.5]] · trị riêng (tăng dần): [0.5 4.5]


In [5]:
# ✅ TỰ KIỂM · chặng 2
assert all(abs(sum(giu_bo(t)) - 5) < 1e-12 for t in range(0, 181, 5))
assert np.allclose(lam2, [0.5, 4.5]) and np.isclose(abs(W2[:, 1] @ np.array([1, 1]) / np.sqrt(2)), 1)
z_C = (P[2] - P.mean(0)) @ (np.array([1, 1]) / np.sqrt(2))
x_hat_C = P.mean(0) + z_C * np.array([1, 1]) / np.sqrt(2)
assert np.allclose(x_hat_C, [6.5, 6.5]) and np.isclose(np.sum((P[2] - x_hat_C) ** 2), 0.5)
print('✅ giữ + bỏ = 5 ở mọi góc; λ = 9/2 và 1/2; PC1 = (1; 1)/√2; C(6; 7) → (6,5; 6,5), lỗi 1/2 (quiz slide 31)')

✅ giữ + bỏ = 5 ở mọi góc; λ = 9/2 và 1/2; PC1 = (1; 1)/√2; C(6; 7) → (6,5; 6,5), lỗi 1/2 (quiz slide 31)


**Câu hỏi:** sửa `C2` để chia cho n − 1 (như scikit-learn). Trị riêng thành bao nhiêu? Hướng có đổi không? (slide 73)

## Chặng 3 · Thuật toán PCA (slide 34–48)

Mười dòng tự viết (slide 41), đối chiếu scikit-learn, rồi lặp lũy thừa trên ma trận 784 × 784 (slide 39).

In [6]:
def pca_fit(X, k):
    mu = X.mean(0)
    Xc = X - mu
    C = Xc.T @ Xc / (len(X) - 1)
    lam, W = np.linalg.eigh(C)     # tăng dần
    idx = np.argsort(lam)[::-1]    # đảo: giảm dần
    return mu, lam[idx], W[:, idx[:k]]
def transform(X, mu, Wk):  return (X - mu) @ Wk
def inverse(Z, mu, Wk):    return Z @ Wk.T + mu

t = time.perf_counter(); mu, lam, W64 = pca_fit(X, 64); print('tự viết:', round(time.perf_counter() - t, 2), 's')
sk = PCA().fit(X)
tot = sk.explained_variance_.sum()
evr = np.cumsum(lam) / lam.sum()
k90 = int(np.searchsorted(evr, 0.90) + 1)
print('λ1, λ2 =', lam[:2].round(4), '· thư viện:', sk.explained_variance_[:2].round(4), '· k90 =', k90, '· solver auto →', sk._fit_svd_solver, '(thuộc tính nội bộ; covariance_eigh có từ scikit-learn 1.5)')

tự viết: 0.62 s


λ1, λ2 = [19.8702 12.1552] · thư viện: [19.8702 12.1552] · k90 = 82 · solver auto → covariance_eigh (thuộc tính nội bộ; covariance_eigh có từ scikit-learn 1.5)


In [7]:
# ✅ TỰ KIỂM · chặng 3 (a): mã tự viết trùng thư viện
tot_ok = lam > 1e-6 * lam[0]
assert np.allclose(lam[tot_ok], sk.explained_variance_[tot_ok], rtol=1e-8)
cosabs = np.abs(np.sum(W64 * sk.components_[:64].T, 0))
assert cosabs.min() > 1 - 1e-8
R1 = inverse(transform(V, mu, W64), mu, W64); R2 = sk.inverse_transform(np.c_[sk.transform(V)[:, :64], np.zeros((len(V), 784 - 64))])
assert np.abs(R1 - R2).max() < 1e-9 and k90 == 82
print('✅', int(tot_ok.sum()), 'trị riêng trùng; |cos| nhỏ nhất =', cosabs.min(), '; ảnh tái tạo trùng; k90 =', k90)
print('dấu bốn PC đầu so với thư viện:', np.sign(np.sum(W64[:, :4] * sk.components_[:4].T, 0)))

✅ 774 trị riêng trùng; |cos| nhỏ nhất = 0.9999999999999987 ; ảnh tái tạo trùng; k90 = 82
dấu bốn PC đầu so với thư viện: [-1. -1. -1. -1.]


In [8]:
# Lặp lũy thừa trên C thật (slide 39): bắt đầu từ vector đều, đếm vòng tới khi 1 − |cos| với PC1 của thư viện < 1e-12 (góc khoảng 1e-4 độ)
Cm = Xc.T @ Xc / (len(X) - 1)
w = np.ones(784) / 28
for vong in range(1, 501):
    w = Cm @ w; w /= np.linalg.norm(w)
    goc = np.degrees(np.arccos(min(1.0, abs(w @ sk.components_[0]))))
    if 1 - abs(w @ sk.components_[0]) < 1e-12:
        break
print('số vòng:', vong, '· góc cuối:', goc, 'độ · wᵀCw =', w @ Cm @ w)

số vòng: 27 · góc cuối: 5.6966603045806464e-05 độ · wᵀCw = 19.870233106231197


In [9]:
# ✅ TỰ KIỂM · chặng 3 (b)
assert vong == 27 and abs(w @ Cm @ w - lam[0]) < 1e-8 * lam[0]
print('✅ 27 vòng, thương Rayleigh = λ1 (slide 39)')

✅ 27 vòng, thương Rayleigh = λ1 (slide 39)


**Câu hỏi:** λ₂/λ₁ ≈ 0,61. Dự đoán: mỗi vòng góc giảm khoảng bao nhiêu lần? Kiểm bằng cách in `goc` ở vài vòng đầu.

## Chặng 4 · PCA trên ảnh thật: lỗi tái tạo, dùng score, giới hạn (slide 49–66)

Mọi số đo trên **validation**. Logistic dùng cùng thiết lập với slide 59 (`C = 1`, `newton-cg`, `tol = 1e-8`); ô logistic trên 784 pixel có thể mất vài chục giây.

In [10]:
Wfull = sk.components_.T              # dùng thư viện cho phần còn lại
Zv = (V - sk.mean_) @ Wfull; Zt = (X - sk.mean_) @ Wfull
def mse_val(k):
    R = sk.mean_ + Zv[:, :k] @ Wfull[:, :k].T
    return np.mean((V - R) ** 2)
for k in [0, 2, 16, 64, 128]:
    print(f'k = {k:3d}: MSE validation {mse_val(k):.4f} · số lưu {(12000 * k + 784 * k + 784) / (12000 * 784):.1%}')

k =   0: MSE validation 0.0867 · số lưu 0.0%
k =   2: MSE validation 0.0466 · số lưu 0.3%
k =  16: MSE validation 0.0207 · số lưu 2.2%
k =  64: MSE validation 0.0106 · số lưu 8.7%
k = 128: MSE validation 0.0065 · số lưu 17.4%


In [11]:
# ✅ TỰ KIỂM · chặng 4 (a)
for k, ref in [(0, 0.08668959363928629), (16, 0.02070311034786636), (64, 0.010561734931280879)]:
    assert abs(mse_val(k) - ref) < 1e-9
print('✅ MSE validation khớp slide 56')

✅ MSE validation khớp slide 56


In [12]:
def logit(A, B):
    m = LogisticRegression(C=1, solver='newton-cg', tol=1e-8, max_iter=1000).fit(A, y)
    return m.score(B, yv)
t = time.perf_counter()
acc64 = logit(Zt[:, :64], Zv[:, :64]); t64 = time.perf_counter() - t
t = time.perf_counter()
accpix = logit(X, V); tpix = time.perf_counter() - t
nn64 = KNeighborsClassifier(1).fit(Zt[:, :64], y).score(Zv[:, :64], yv)
se = np.sqrt(accpix * (1 - accpix) / len(yv))
print(f'logistic k = 64: {acc64:.4f} ({t64:.1f} s) · pixel: {accpix:.4f} ({tpix:.1f} s) · sai số chuẩn ≈ {se:.4f} · 1-NN k = 64: {nn64:.4f}')

logistic k = 64: 0.8370 (3.2 s) · pixel: 0.8337 (21.3 s) · sai số chuẩn ≈ 0.0068 · 1-NN k = 64: 0.8183


In [13]:
# ✅ TỰ KIỂM · chặng 4 (b)
assert abs(acc64 - 0.837) < 1e-9 and abs(nn64 - 0.8183333333333334) < 1e-9
assert abs(accpix - 0.8336666666666667) < 0.002   # tối ưu hoá có thể lệch một hai ảnh tuỳ phiên bản thư viện
print('✅ k = 64 ngang pixel (chênh', round(acc64 - accpix, 4), '< một sai số chuẩn', round(se, 4), ')')

✅ k = 64 ngang pixel (chênh 0.0033 < một sai số chuẩn 0.0068 )


In [14]:
# Dịch ảnh validation sang phải 2 pixel (slide 63)
Vs = np.zeros_like(V).reshape(-1, 28, 28); Vs[:, :, 2:] = V.reshape(-1, 28, 28)[:, :, :-2]; Vs = Vs.reshape(-1, 784)
Zs = (Vs - sk.mean_) @ Wfull[:, :64]
ty_so = np.mean((Vs - (sk.mean_ + Zs @ Wfull[:, :64].T)) ** 2) / mse_val(64)
print('lỗi tái tạo tăng', round(ty_so, 3), 'lần')

lỗi tái tạo tăng 1.704 lần


In [15]:
# ✅ TỰ KIỂM · chặng 4 (c)
assert abs(ty_so - 1.703781012) < 1e-6
print('✅ dịch 2 pixel: lỗi tăng khoảng 1,7 lần (slide 63)')

✅ dịch 2 pixel: lỗi tăng khoảng 1,7 lần (slide 63)


**Câu hỏi cuối:** (1) Với ràng buộc “lưu không quá 5% số con số”, bạn chọn k nào cho tìm ảnh giống? Dẫn số đo. (2) Viết hai câu báo cáo kết quả logistic k = 64 so với pixel, có sai số chuẩn và không nói quá.